Starter Notebook Header Block

In [1]:
# ===============================
# Phase 3: Retrieve-and-Rerank Setup
# Blood Donor Chatbot
# ===============================


# Imports
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer, CrossEncoder, util


# ===============================
# Load datasets
# ===============================

train_df = pd.read_csv("train.csv")
validation_df = pd.read_csv("validation.csv")

print("Train shape:", train_df.shape)
print("Validation shape:", validation_df.shape)


# ===============================
# Prepare question and answer lists
# ===============================

train_questions = train_df["Question"].tolist()
train_answers = train_df["Answer"].tolist()

validation_questions = validation_df["Question"].tolist()
validation_answers = validation_df["Answer"].tolist()


# ===============================
# Load semantic embedding model (bi-encoder)
# ===============================

bi_encoder = SentenceTransformer("all-MiniLM-L6-v2")

print("Bi-encoder loaded successfully.")


# ===============================
# Encode all training questions
# ===============================

question_embeddings = bi_encoder.encode(
    train_questions,
    convert_to_tensor=True,
    show_progress_bar=True
)

print("Training embeddings created.")
print("Embedding shape:", question_embeddings.shape)


# ===============================
# Load reranker model (cross-encoder)
# ===============================

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L6-v2")

print("Cross-encoder reranker loaded successfully.")


# ===============================
# Setup confirmation
# ===============================

print("\nPhase 3 environment ready.")
print("Total training questions indexed:", len(train_questions))
print("Total validation questions ready:", len(validation_questions))

Train shape: (2700, 2)
Validation shape: (338, 2)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Bi-encoder loaded successfully.


Batches:   0%|          | 0/85 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Cross-encoder reranker loaded successfully.

Phase 3 environment ready.
Total training questions indexed: 2700
Total validation questions ready: 338


Define retrieve-and-rerank function

In [2]:
def retrieve_and_rerank(user_query, questions, answers, embeddings, bi_encoder_model, reranker_model, top_k=5):
    # Step 1: retrieve top-k with bi-encoder
    query_embedding = bi_encoder_model.encode(user_query, convert_to_tensor=True)
    similarities = util.cos_sim(query_embedding, embeddings)[0]
    
    top_results = similarities.topk(k=top_k)
    top_indices = top_results.indices.tolist()
    top_scores = top_results.values.tolist()
    
    candidate_questions = [questions[idx] for idx in top_indices]
    candidate_answers = [answers[idx] for idx in top_indices]
    
    # Step 2: rerank top-k with cross-encoder
    sentence_pairs = [[user_query, candidate_question] for candidate_question in candidate_questions]
    rerank_scores = reranker_model.predict(sentence_pairs)
    
    best_rerank_idx = int(np.argmax(rerank_scores))
    
    return {
        "user_query": user_query,
        "matched_question": candidate_questions[best_rerank_idx],
        "retrieved_answer": candidate_answers[best_rerank_idx],
        "bi_encoder_score": top_scores[best_rerank_idx],
        "reranker_score": float(rerank_scores[best_rerank_idx]),
        "all_candidates": pd.DataFrame({
            "candidate_question": candidate_questions,
            "candidate_answer": candidate_answers,
            "bi_encoder_score": top_scores,
            "reranker_score": rerank_scores
        }).sort_values("reranker_score", ascending=False).reset_index(drop=True)
    }

Testing the reranked chatbot on one query

In [4]:
test_result = retrieve_and_rerank(
    user_query="Can I donate blood if I have a cold?",
    questions=train_questions,
    answers=train_answers,
    embeddings=question_embeddings,
    bi_encoder_model=bi_encoder,
    reranker_model=reranker,
    top_k=5
)

print("User Query:", test_result["user_query"])
print("Matched Question:", test_result["matched_question"])
print("Retrieved Answer:", test_result["retrieved_answer"])
print("Bi-encoder Score:", round(test_result["bi_encoder_score"], 4))
print("Reranker Score:", round(test_result["reranker_score"], 4))

User Query: Can I donate blood if I have a cold?
Matched Question: Can individuals who have recovered from certain medical conditions or surgeries donate blood, and if so, are there any restrictions?
Retrieved Answer: Individuals who have recovered from specific medical conditions or surgeries may be eligible to donate blood, depending on factors such as the nature of the condition, treatment received, and overall health status, subject to assessment by healthcare professionals and blood donation centers.
Bi-encoder Score: 0.6432
Reranker Score: -2.4361


View all reranked candidates for that query

In [5]:
test_result["all_candidates"]

,candidate_question,candidate_answer,bi_encoder_score,reranker_score
0,Can individuals who have recovered from certai...,Individuals who have recovered from specific m...,0.643249,-2.436144
1,How can blood donation impact individuals with...,"Blood donation may help manage iron levels, wh...",0.635241,-3.226397
2,Can individuals who have recently received vac...,"In most cases, individuals who have recently r...",0.665911,-3.402622
3,Can individuals who have recently undergone bl...,"In many cases, individuals who have recently u...",0.634348,-3.658063
4,How can blood donation support individuals wit...,Blood donation supports individuals with chron...,0.641170,-5.302936


Running reranking on the whole validation set

In [6]:
reranked_results = []

for true_question, true_answer in zip(validation_questions, validation_answers):
    output = retrieve_and_rerank(
        user_query=true_question,
        questions=train_questions,
        answers=train_answers,
        embeddings=question_embeddings,
        bi_encoder_model=bi_encoder,
        reranker_model=reranker,
        top_k=5
    )
    
    reranked_results.append({
        "validation_question": true_question,
        "expected_answer": true_answer,
        "matched_question": output["matched_question"],
        "retrieved_answer": output["retrieved_answer"],
        "bi_encoder_score": output["bi_encoder_score"],
        "reranker_score": output["reranker_score"]
    })

reranked_results_df = pd.DataFrame(reranked_results)

print("Reranked validation evaluation completed.")
reranked_results_df.head()

Reranked validation evaluation completed.


,validation_question,expected_answer,matched_question,retrieved_answer,bi_encoder_score,reranker_score
0,What impact do intergenerational donation even...,"Intergenerational donation events, such as fam...",What impact do blood donor recognition program...,"Blood donor recognition programs, appreciation...",0.715886,2.358362
1,How do blood drive hosts collaborate with acad...,Blood drive hosts collaborate with academic in...,How do blood drive hosts collaborate with scho...,"Blood drive hosts collaborate with schools, co...",0.836275,5.290590
2,What factors contribute to disparities in bloo...,Disparities in blood donation rates among diff...,What factors contribute to disparities in bloo...,Disparities in blood donation rates among raci...,0.948832,7.050753
3,What role can celebrity endorsements and partn...,Celebrity endorsements and influencer partners...,What role can celebrity endorsements and influ...,Celebrity endorsements and influencer partners...,0.986526,8.833264
4,How does blood donation support families of pa...,Blood donation supports families of patients b...,How can donating blood promote intergeneration...,Blood donation can encourage families to parti...,0.806792,2.070389


Compute reranked semantic accuracy

In [7]:
reranked_similarity_scores = []

for _, row in reranked_results_df.iterrows():
    val_q = row["validation_question"]
    matched_q = row["matched_question"]
    
    val_emb = bi_encoder.encode(val_q, convert_to_tensor=True)
    matched_emb = bi_encoder.encode(matched_q, convert_to_tensor=True)
    
    sim_score = util.cos_sim(val_emb, matched_emb).item()
    reranked_similarity_scores.append(sim_score)

reranked_results_df["semantic_similarity"] = reranked_similarity_scores

print(reranked_results_df["semantic_similarity"].describe())

count    338.000000
mean       0.879726
std        0.066658
min        0.619897
25%        0.847966
50%        0.887120
75%        0.928802
max        0.998680
Name: semantic_similarity, dtype: float64


Threshold-based reranked accuracy

In [9]:
semantic_accuracy_085 = (reranked_results_df["semantic_similarity"] >= 0.85).mean() * 100
semantic_accuracy_090 = (reranked_results_df["semantic_similarity"] >= 0.90).mean() * 100

print("Reranked Semantic Retrieval Accuracy (>= 0.85):", round(semantic_accuracy_085, 2), "%")
print("Reranked Strict Semantic Retrieval Accuracy (>= 0.90):", round(semantic_accuracy_090, 2), "%")

Reranked Semantic Retrieval Accuracy (>= 0.85): 72.78 %
Reranked Strict Semantic Retrieval Accuracy (>= 0.90): 41.72 %


Compare baseline vs reranked system

In [10]:
comparison_df = pd.DataFrame({
    "Metric": [
        "Semantic Accuracy >= 0.85",
        "Strict Semantic Accuracy >= 0.90"
    ],
    "Baseline": [
        81.07,
        51.18
    ],
    "Reranked": [
        round(semantic_accuracy_085, 2),
        round(semantic_accuracy_090, 2)
    ]
})

comparison_df["Improvement"] = comparison_df["Reranked"] - comparison_df["Baseline"]
comparison_df

,Metric,Baseline,Reranked,Improvement
0,Semantic Accuracy >= 0.85,81.07,72.78,-8.29
1,Strict Semantic Accuracy >= 0.90,51.18,41.72,-9.46


Inspect weakest reranked cases

In [11]:
reranked_worst_cases = reranked_results_df.sort_values("semantic_similarity", ascending=True).head(20)

reranked_worst_cases[[
    "validation_question",
    "matched_question",
    "retrieved_answer",
    "bi_encoder_score",
    "reranker_score",
    "semantic_similarity"
]]

,validation_question,matched_question,retrieved_answer,bi_encoder_score,reranker_score,semantic_similarity
33,What factors do bottom-up plans consider in es...,What is the focus of bottom-up planning proces...,Bottom-up plans focus on establishing potentia...,0.619897,0.155653,0.619897
319,What innovative approaches are being explored ...,What are the potential challenges of implement...,Challenges include the availability of equipme...,0.670546,-1.272735,0.670546
85,What are the implications of a reactive screen...,How do national guidelines address the handlin...,National guidelines provide guidance on the ha...,0.690089,3.124325,0.690089
57,How can blood donation support individuals wit...,How can blood donation support individuals wit...,Blood donation can provide a sense of purpose ...,0.708591,2.238811,0.708591
46,What factors influenced the timing and frequen...,In what manner were blood transfusions adminis...,Blood transfusions were administered intraveno...,0.710674,-6.735310,0.710674
131,How can individuals benefit from quitting smok...,How can blood donation benefit individuals wit...,Blood donation may help regulate ferrous level...,0.712849,-1.709760,0.712849
219,What factors contribute to demographic dispari...,How do donor deferral policies impact donation...,Donor deferral policies can impact donation ra...,0.713977,2.460072,0.713977
0,What impact do intergenerational donation even...,What impact do blood donor recognition program...,"Blood donor recognition programs, appreciation...",0.715886,2.358362,0.715886
214,What strategies are employed to ensure inclusi...,What strategies are employed to promote long-t...,Strategies employed to promote long-term engag...,0.720043,3.764457,0.720043
10,How has the World Health Organization addresse...,What initiatives has Tanzania's national blood...,Tanzania's national blood supplier has impleme...,0.727678,-2.255287,0.727677


Save Phase 3 results

In [12]:
reranked_results_df.to_csv("validation_results_reranked.csv", index=False)
reranked_worst_cases.to_csv("validation_worst_cases_reranked.csv", index=False)
comparison_df.to_csv("baseline_vs_reranked_comparison.csv", index=False)

print("Saved Phase 3 reranking results.")

Saved Phase 3 reranking results.
